# AI Travel Assistant & Multi-Tool Agent

##   Setup , Imports & API Configuration

In [ ]:
# imports
import os
import json
import sqlite3
import base64
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Custom tools and DB helper from local modules
from tools import get_live_weather, get_live_attractions, get_live_currency
from db import setup_airbnb_database, get_airbnb_listings, DB

In [ ]:
# Initialization of API Keys, Models, and OpenAI SDK Clients
load_dotenv(override=True)

# OpenRouter Client for General AI & Audio TTS
openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# Gemini Client via OpenAI Compatible API for Image Generation
gemini = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GEMINI_API_KEY")
)

# Model Definitions
MODEL = "openrouter/free"
TTS_MODEL = "openai/tts-1"
IMAGE_MODEL = "google/gemini-3.1-flash-lite-image"

# Initialize SQLite travel/airbnb database
setup_airbnb_database()

##   System Prompt — Wanderer AI Persona

In [ ]:
# System Prompt Definition
system_message = """
You are a helpful AI travel assistant for an agency called Wanderer AI.
Give concise, courteous, accurate answers.
Always use available tools to lookup flight prices, live weather, attractions, currency rates, or Airbnb listings.
If you don't know the answer or lack data, say so clearly.
"""

## SQLite Database — Test Airbnb Listings

In [ ]:
# Test direct database tool functions
print(get_airbnb_listings("Paris", max_budget=150))

## Tool Schemas — Function Calling Definitions

In [ ]:
# Tool Schemas for OpenAI Function Calling
airbnb_function = {
    "name": "get_airbnb_listings",
    "description": "Search SQLite database for Airbnb accommodations by city and optional max budget.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The target city name"},
            "max_budget": {"type": "number", "description": "Maximum night price limit in USD"}
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

weather_function = {
    "name": "get_live_weather",
    "description": "Fetch real-time live weather and 5-day forecast for any city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The city name"}
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

currency_function = {
    "name": "get_live_currency",
    "description": "Get live exchange rates for a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The target city"},
            "from_currency": {"type": "string", "description": "Base currency code, default USD"}
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

attractions_function = {
    "name": "get_live_attractions",
    "description": "Fetch top tourist attractions for a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The target city"}
        },
        "required": ["city"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": airbnb_function},
    {"type": "function", "function": weather_function},
    {"type": "function", "function": currency_function},
    {"type": "function", "function": attractions_function}
]

## Tool Handler — Dispatch & Execute Tool Calls

In [ ]:
def handle_tool_calls(message):
    """Dispatches model tool calls to target functions and collects responses."""
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        fn_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"Executing tool call: {fn_name} with args: {args}", flush=True)
        
        content = ""
        if fn_name == "get_airbnb_listings":
            city = args.get('city')
            if city:
                cities.append(city)
            budget = args.get('max_budget')
            content = get_airbnb_listings(city, budget)
        elif fn_name == "get_live_weather":
            city = args.get('city')
            if city:
                cities.append(city)
            content = get_live_weather(city)
        elif fn_name == "get_live_currency":
            city = args.get('city')
            if city:
                cities.append(city)
            base = args.get('from_currency', 'USD')
            content = get_live_currency(city, base)
        elif fn_name == "get_live_attractions":
            city = args.get('city')
            if city:
                cities.append(city)
            content = get_live_attractions(city)
            
        responses.append({
            "role": "tool",
            "content": str(content),
            "tool_call_id": tool_call.id
        })
    return responses, cities

## Basic Chat Function — Tool-Calling Loop

In [ ]:
def chat(message, history):
    """Standard chat function supporting while loop tool calls."""
    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_msgs + [{"role": "user", "content": message}]
    response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses, _ = handle_tool_calls(msg)
        messages.append(msg)
        messages.extend(tool_responses)
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

## Image Generation — Destination Travel Poster (Pollinations AI)

In [ ]:
def artist(city):
    """Generates a vibrant travel poster using Pollinations AI (100% Free, No API Key needed)."""
    try:
        from urllib.parse import quote
        import requests

        print(f"Generating travel image for {city} via Pollinations AI...", flush=True)
        prompt = f"An image representing a vacation in {city}, showing tourist spots in a vibrant pop-art style"
        encoded_prompt = quote(prompt)

        # Pollinations AI Free Image Endpoint
        image_url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=1024&height=1024&nologo=true"

        resp = requests.get(image_url, timeout=15)
        resp.raise_for_status()
        return Image.open(BytesIO(resp.content))
    except Exception as e:
        print(f"Image generation error/skipped: {e}", flush=True)
        return None



## Text-to-Speech — Voice Response (Google TTS)

In [ ]:
def talker(message):
    """Generates voice speech audio using Google TTS (100% Free)."""
    try:
        import re
        from urllib.parse import quote
        import requests

        # 1. Clean markdown formatting (*, #, _, \n, etc.) into plain text
        plain_text = re.sub(r"[\*\#\_\`\~\-\n\r]", " ", message)
        plain_text = re.sub(r"\s+", " ", plain_text).strip()[:200]

        if not plain_text:
            return None

        # 2. Encode clean text into Google TTS URL
        url = f"https://translate.google.com/translate_tts?ie=UTF-8&q={quote(plain_text)}&tl=en&client=tw-ob"
        headers = {"User-Agent": "Mozilla/5.0"}

        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
        return resp.content
    except Exception as e:
        print(f"TTS speech error/skipped: {e}", flush=True)
        return None


## Multimodal Chat — Audio + Image + Tools Combined

In [ ]:
def chat_multimodal(history):
    """Full multimodal chat routine returning updated history, audio speech, and destination image."""
    history_msgs = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history_msgs
    
    response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    all_cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        tool_responses, cities = handle_tool_calls(msg)
        all_cities.extend(cities)
        messages.append(msg)
        messages.extend(tool_responses)
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history_msgs.append({"role": "assistant", "content": reply})

    voice = talker(reply)
    if all_cities:
        image = artist(all_cities[0])
    
    return history_msgs, voice, image

## Chatbot Message Handler & Gradio UI Launch 

In [ ]:
def put_message_in_chatbot(message, history):
    """Clears text input box and immediately appends user message to chatbot history."""
    return "", history + [{"role": "user", "content": message}]

In [ ]:
with gr.Blocks(title="Wanderer AI Travel Assistant") as ui:
    gr.Markdown("# ✈️ Wanderer AI Travel Assistant")
    gr.Markdown("Ask about Airbnb listings, live weather, currency exchange, or tourist attractions!")
    
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False, label="Destination Visualizer")
    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="Voice Response")
    with gr.Row():
        message = gr.Textbox(label="Chat with Wanderer AI Assistant:", placeholder="e.g., What is the weather and Airbnb price in Paris?")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat_multimodal, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=False)